# 03. Deep Learning: Convolutional Neural Network (CNN) Digit Classifier

### Overview
This notebook designs, trains, evaluates, and exports the Convolutional Neural Network (CNN) used to recognize Sudoku digits (`1`–`9` and `empty`).

### Model Architecture
```
Input (28, 28, 1)
  │
  ├── Data Augmentation (Rotation, Zoom, Translation)
  ├── Conv2D(32, 3x3) + BatchNorm + ReLU + MaxPool(2x2)
  ├── Conv2D(64, 3x3) + BatchNorm + ReLU + MaxPool(2x2)
  ├── Conv2D(128, 3x3) + BatchNorm + ReLU + MaxPool(2x2)
  ├── Flatten
  ├── Dense(128, ReLU) + Dropout(0.4)
  └── Dense(10, Softmax)
```

In [ ]:
import os
import sys
from pathlib import Path

def find_root() -> Path:
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / "models" / "digit_classifier.keras").exists():
            return parent
    return Path(os.path.abspath("..")).resolve()

PROJECT_ROOT = find_root()
MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data" / "digits"

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Reproducibility seed
tf.keras.utils.set_random_seed(42)

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPUs Available: {tf.config.list_physical_devices('GPU')}")

## 1. Dataset Loading & Partitioning
We load the dataset using `image_dataset_from_directory` with an 80/20 train/validation split.

In [ ]:
IMG_SIZE = (28, 28)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    color_mode="grayscale",
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    color_mode="grayscale",
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Class labels:", class_names)

## 2. Pipeline Performance Optimization
Normalize pixel values from $[0, 255]$ to $[0, 1]$ and optimize memory transfer with `.cache()` and `.prefetch(AUTOTUNE)`.

In [ ]:
normalization_layer = layers.Rescaling(1.0 / 255.0)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(buffer_size=AUTOTUNE)

## 3. Network Architecture & Regularization
We define a 3-stage feature extractor with Batch Normalization for gradient stability and Dropout to prevent overfitting.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),
    layers.RandomTranslation(0.05, 0.05),
], name="data_augmentation")

num_classes = len(class_names)

model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    data_augmentation,

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
], name="sudoku_digit_cnn")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 4. Model Training with Early Stopping
We train for up to 60 epochs with an `EarlyStopping` callback configured with `restore_best_weights=True`.

In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-5,
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stop, reduce_lr]
)

## 5. Training Curves & Convergence Diagnostics
Visualize validation accuracy and cross-entropy loss across epochs.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy", lw=2)
plt.plot(history.history["val_accuracy"], label="Val Accuracy", lw=2)
plt.title("Classification Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss", lw=2)
plt.plot(history.history["val_loss"], label="Val Loss", lw=2)
plt.title("Cross-Entropy Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Confusion Matrix & Classification Report
Evaluate class-by-class precision, recall, and potential digit confusions (e.g., distinguishing 1 vs 7, or empty vs noise).

In [ ]:
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(cmap="Blues", ax=ax, xticks_rotation=45)
plt.title("Validation Confusion Matrix")
plt.show()

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

## 7. Model Serialization
Export the trained model weights in Keras v3 format (`.keras`).

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
save_target = MODELS_DIR / "digit_classifier.keras"

model.save(str(save_target))
print(f"Model saved successfully to: {save_target}")